## 4. LSTM Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import LSTMPredictor, HORIZONS

MODEL_NAME = 'LSTM'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'lstm'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'lstm_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'lstm_val_stability_summary.json'

In [3]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

val_participant_ids = load_participant_ids('val')
test_participant_ids = load_participant_ids('test')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [4]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = LSTMPredictor()

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'lstm' /f'lstm_seed{seed}',
        verbose=False,
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'lstm_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")


Validation results — LSTM | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: LSTMPredictor  params: 207,749


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.09 | MAE 7.43 | gRMSE 11.81 | Zone A 96.60% | A+B 99.70%
 30 min | RMSE 17.63 | MAE 11.73 | gRMSE 19.68 | Zone A 89.72% | A+B 99.34%
 60 min | RMSE 25.06 | MAE 16.75 | gRMSE 29.53 | Zone A 81.59% | A+B 98.72%
 90 min | RMSE 29.41 | MAE 19.88 | gRMSE 35.41 | Zone A 76.11% | A+B 98.17%
120 min | RMSE 32.05 | MAE 22.12 | gRMSE 38.98 | Zone A 72.12% | A+B 97.86%

--------------------------------------------------------------------------------
Validation results — LSTM | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: LSTMPredictor  params: 207,749


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.23 | MAE 7.67 | gRMSE 12.44 | Zone A 96.64% | A+B 99.70%
 30 min | RMSE 17.61 | MAE 11.93 | gRMSE 19.94 | Zone A 89.44% | A+B 99.35%
 60 min | RMSE 25.01 | MAE 17.04 | gRMSE 29.59 | Zone A 81.11% | A+B 98.67%
 90 min | RMSE 29.31 | MAE 20.16 | gRMSE 35.38 | Zone A 75.82% | A+B 98.22%
120 min | RMSE 31.89 | MAE 22.30 | gRMSE 38.85 | Zone A 72.30% | A+B 97.85%

--------------------------------------------------------------------------------
Validation results — LSTM | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: LSTMPredictor  params: 207,749


...

 15 min | RMSE 11.02 | MAE 7.42 | gRMSE 11.92 | Zone A 96.72% | A+B 99.66%
 30 min | RMSE 17.62 | MAE 11.80 | gRMSE 19.81 | Zone A 89.64% | A+B 99.30%
 60 min | RMSE 25.07 | MAE 16.86 | gRMSE 29.61 | Zone A 81.38% | A+B 98.69%
 90 min | RMSE 29.46 | MAE 20.13 | gRMSE 35.46 | Zone A 75.84% | A+B 98.17%
120 min | RMSE 32.09 | MAE 22.35 | gRMSE 38.98 | Zone A 72.09% | A+B 97.83%

--------------------------------------------------------------------------------
Validation results — LSTM | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: LSTMPredictor  params: 207,749


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.09 | MAE 7.45 | gRMSE 12.18 | Zone A 96.80% | A+B 99.67%
 30 min | RMSE 17.59 | MAE 11.72 | gRMSE 19.94 | Zone A 89.82% | A+B 99.32%
 60 min | RMSE 24.95 | MAE 16.62 | gRMSE 29.68 | Zone A 81.62% | A+B 98.69%
 90 min | RMSE 29.19 | MAE 19.73 | gRMSE 35.41 | Zone A 76.46% | A+B 98.18%
120 min | RMSE 31.75 | MAE 21.83 | gRMSE 38.87 | Zone A 72.85% | A+B 97.87%

--------------------------------------------------------------------------------
Validation results — LSTM | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: LSTMPredictor  params: 207,749


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.95 | MAE 7.41 | gRMSE 11.96 | Zone A 96.79% | A+B 99.68%
 30 min | RMSE 17.46 | MAE 11.70 | gRMSE 19.71 | Zone A 89.87% | A+B 99.37%
 60 min | RMSE 24.91 | MAE 16.65 | gRMSE 29.46 | Zone A 81.58% | A+B 98.67%
 90 min | RMSE 29.19 | MAE 19.77 | gRMSE 35.21 | Zone A 76.61% | A+B 98.14%
120 min | RMSE 31.69 | MAE 21.80 | gRMSE 38.58 | Zone A 72.94% | A+B 97.87%

--------------------------------------------------------------------------------


In [9]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [10]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [11]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== LSTM validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.04 +/- 0.10 (2.47) mg/dL   MAE = 7.45 +/- 0.11 (1.66)   gRMSE = 12.04 +/- 0.20 (3.39)   Zone A = 96.78 +/- 0.13 (3.08)%   Zone A+B = 99.69 +/- 0.03 (0.60)%
 30-min:  RMSE = 17.55 +/- 0.09 (3.85) mg/dL   MAE = 11.73 +/- 0.14 (2.47)   gRMSE = 19.81 +/- 0.13 (5.78)   Zone A = 89.77 +/- 0.21 (5.65)%   Zone A+B = 99.34 +/- 0.02 (0.96)%
 60-min:  RMSE = 24.98 +/- 0.09 (7.39) mg/dL   MAE = 16.68 +/- 0.21 (4.70)   gRMSE = 29.59 +/- 0.11 (11.19)   Zone A = 81.59 +/- 0.32 (5.36)%   Zone A+B = 98.68 +/- 0.02 (1.44)%
 90-min:  RMSE = 29.29 +/- 0.12 (10.30) mg/dL   MAE = 19.81 +/- 0.23 (6.77)   gRMSE = 35.41 +/- 0.14 (15.33)   Zone A = 76.39 +/- 0.48 (7.26)%   Zone A+B = 98.17 +/- 0.03 (2.03)%
120-min:  RMSE = 31.86 +/- 0.17 (12.00) mg/dL   MAE = 21.93 +/- 0.29 (7.99)   gRMSE = 38.86 +/- 0.19 (17.96)   Zone A = 72.70 +/- 0.46 (8.08)%   Zone A+B = 97.82 +/- 0.05 (2.59)%


In [12]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/lstm/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/lstm/lstm_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/lstm/lstm_stability_summary.json
